### **Chapter 9.2: Willems' Fundamental Lemma**

This notebook gives a direct numerical showcase of the key result behind DeePC:
for a controllable LTI system, one sufficiently persistently exciting measured trajectory can span **all finite trajectories of a chosen length**.

We demonstrate three things:

1. an unseen Mountain-Car trajectory can be reconstructed from Hankel columns;
2. reconstruction fails when the offline input is not persistently exciting;
3. the transition can be visualized by sweeping the amount of offline data.

As in the rest of the chapter the measured output is the position only, $y = p$; the velocity is hidden. The main example uses the flat terrain (`case = 1`). At the end, we repeat the same idea for the constant-slope case after shifting to equilibrium coordinates.

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath(".."))
from utils.env import *
from ex9_DeePC.deepc_utils import *

### **Part 1: Collect One Persistently Exciting Trajectory**

For a trajectory length $L$, Willems' fundamental lemma requires the offline input to be persistently exciting of order $L+n$, where $n$ is the system order.

For the Mountain Car linear case,

$$
n=2, \qquad m=1, \qquad y = p .
$$

We first collect one random open-loop trajectory and check the rank of $H_{L+n}(u^d)$.

In [ ]:
case = 1
freq = 20
dt = 1.0 / freq
L = 12
n_data = 180

initial_state = np.array([-0.5, 0.0])
target_state = np.array([0.6, 0.0])

env = Env(case, initial_state, target_state, input_lbs=-1.0, input_ubs=1.0)
dynamics = Dynamics(env)

u_data, y_data = collect_deepc_data(
    env,
    dynamics,
    freq=freq,
    n_samples=n_data,
    excitation_amplitude=0.7,
    initial_state=env.target_state,
    seed=1,
    output_indices=[0],
)

pe_order = L + dynamics.dim_states
rank, required_rank = persistent_excitation_rank(
    u_data,
    pe_order,
    input_offset=np.atleast_1d(dynamics.get_equilibrium_input(env.target_state)),
)

print(f"PE order: {pe_order}")
print(f"rank(H_{pe_order}(u)) = {rank} / {required_rank}")
print("Persistently exciting:", rank == required_rank)

### **Part 2: Reconstruct an Unseen Trajectory**

Now generate a second trajectory with a different initial condition and a different input sequence. This trajectory was never included in the offline data.

We solve only for the coefficient vector $g$:

$$
\min_g \left\|
\begin{bmatrix}H_L(u^d)\\H_L(y^d)\end{bmatrix}g
-
\begin{bmatrix}u^{\rm test}\\y^{\rm test}\end{bmatrix}
\right\|_2^2.
$$

If the assumptions of the fundamental lemma hold, the residual should be at numerical precision.

In [ ]:
def rollout(dynamics, x0, u_sequence, dt):
    x = np.asarray(x0, dtype=float).copy()
    states = [x.copy()]
    for u in np.asarray(u_sequence):
        x = dynamics.one_step_forward(x, np.asarray(u).reshape(-1), dt)
        states.append(x.copy())
    return np.asarray(states)

rng = np.random.default_rng(12)
u_test = rng.uniform(-0.55, 0.55, size=(L, 1))
x_test = rollout(dynamics, np.array([-0.35, 0.18]), u_test, dt)
y_test = x_test[:, [0]]   # only the position is measured

u_eq = np.atleast_1d(dynamics.get_equilibrium_input(env.target_state))
_, u_rec, y_rec, rel_error = reconstruct_trajectory(
    u_data,
    y_data,
    u_test,
    y_test,
    length=L,
    input_offset=u_eq,
    output_offset=np.array([env.target_position]),
)

print(f"Relative reconstruction error: {rel_error:.3e}")

In [ ]:
t = np.arange(L) / freq
fig, ax = plt.subplots(2, 1, figsize=(9, 6), sharex=True)

ax[0].plot(t, u_test[:, 0], label="test")
ax[0].plot(t, u_rec[:, 0], "--", label="Hankel reconstruction")
ax[0].set_ylabel("input")
ax[0].legend()

ax[1].plot(t, y_test[:L, 0], label="test")
ax[1].plot(t, y_rec[:, 0], "--", label="Hankel reconstruction")
ax[1].set_ylabel("position")
ax[1].set_xlabel("Time (s)")

fig.suptitle("Fundamental Lemma: reconstruction of an unseen trajectory")
plt.tight_layout()
plt.show()

<blockquote style="padding: 18px 20px; margin: 1.2em 0; background: rgba(56, 139, 253, 0.12); border-left: 4px solid rgba(56, 139, 253, 0.85); border-radius: 6px; color: inherit !important;">

##### **Takeaway 1: One sufficiently rich trajectory spans all finite LTI trajectories**

The Hankel matrix is not merely fitting the particular measured experiment. Under persistent excitation, its column space represents the finite behavior of the LTI system.
</blockquote>

### **Part 3: What Happens Without Persistent Excitation?**

The online initialization trajectory itself does **not** need to be persistently exciting. Persistent excitation is a requirement on the **offline dataset used to build the Hankel matrix**.

To make the failure obvious, collect an offline dataset with constant zero input. Its input Hankel matrix is rank deficient, and it cannot represent the new test trajectory.

In [ ]:
u_bad = np.zeros((n_data, 1))
y_bad = rollout(dynamics, env.target_state, u_bad, dt)[:, [0]]

rank_bad, required_bad = persistent_excitation_rank(u_bad, pe_order)
_, _, _, rel_error_bad = reconstruct_trajectory(
    u_bad,
    y_bad,
    u_test,
    y_test,
    length=L,
    input_offset=np.zeros(1),
    output_offset=np.array([env.target_position]),
)

print(f"rank(H_{pe_order}(u_bad)) = {rank_bad} / {required_bad}")
print(f"Relative reconstruction error with non-PE data: {rel_error_bad:.3e}")

### **Part 4: Data Length Sweep**

For a scalar input, a necessary dimensional requirement for $H_r(u)$ to have full row rank is that it has at least as many columns as rows. With $r=L+n$, this gives the familiar sufficient data-length scaling

$$
T \ge (m+1)(L+n)-1.
$$

The following sweep plots both the Hankel rank and the unseen-trajectory reconstruction error as the available offline dataset grows.

In [ ]:
lengths = np.arange(pe_order + 2, n_data + 1, 4)
ranks = []
errors = []

for T in lengths:
    try:
        rank_T, _ = persistent_excitation_rank(u_data[:T], pe_order, input_offset=u_eq)
        _, _, _, err_T = reconstruct_trajectory(
            u_data[:T],
            y_data[:T + 1],
            u_test,
            y_test,
            length=L,
            input_offset=u_eq,
            output_offset=np.array([env.target_position]),
        )
    except ValueError:
        rank_T = np.nan
        err_T = np.nan
    ranks.append(rank_T)
    errors.append(err_T)

T_sufficient = (dynamics.dim_inputs + 1) * (L + dynamics.dim_states) - 1

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(lengths, ranks, marker="o", markersize=3)
ax[0].axhline(required_rank, linestyle="--", label="full row rank")
ax[0].axvline(T_sufficient, linestyle=":", label="sufficient length bound")
ax[0].set_xlabel("Offline input samples T")
ax[0].set_ylabel("rank")
ax[0].set_title("Persistent-excitation rank")
ax[0].legend()

ax[1].semilogy(lengths, np.maximum(errors, 1e-16), marker="o", markersize=3)
ax[1].axvline(T_sufficient, linestyle=":", label="sufficient length bound")
ax[1].set_xlabel("Offline input samples T")
ax[1].set_ylabel("relative reconstruction error")
ax[1].set_title("Behavior reconstruction")
ax[1].legend()

plt.tight_layout()
plt.show()

### **Part 5: Constant Slope = Affine Dynamics + Equilibrium Shift**

The constant-slope Mountain Car is affine in the original coordinates because gravity contributes a constant acceleration. Around the target equilibrium,

$$
\tilde x=x-x_{\rm ref},\qquad \tilde u=u-u_{\rm eq},
$$

and the deviation dynamics are LTI. Therefore the same fundamental-lemma experiment can be repeated in shifted coordinates.

In [ ]:
env_slope = Env(
    2,
    initial_state,
    target_state,
    input_lbs=-2.5,
    input_ubs=2.5,
)
dynamics_slope = Dynamics(env_slope)
u_eq_slope = np.atleast_1d(dynamics_slope.get_equilibrium_input(env_slope.target_state))

u_data_slope, y_data_slope = collect_deepc_data(
    env_slope,
    dynamics_slope,
    freq=freq,
    n_samples=n_data,
    excitation_amplitude=0.7,
    initial_state=env_slope.target_state,
    seed=3,
    output_indices=[0],
)

rng = np.random.default_rng(4)
u_test_slope = u_eq_slope + rng.uniform(-0.5, 0.5, size=(L, 1))
y_test_slope = rollout(
    dynamics_slope,
    env_slope.target_state + np.array([-0.25, 0.12]),
    u_test_slope,
    dt,
)[:, [0]]

rank_slope, required_slope = persistent_excitation_rank(
    u_data_slope, pe_order, input_offset=u_eq_slope
)
_, _, _, error_slope = reconstruct_trajectory(
    u_data_slope,
    y_data_slope,
    u_test_slope,
    y_test_slope,
    length=L,
    input_offset=u_eq_slope,
    output_offset=np.array([env_slope.target_position]),
)

print("equilibrium input:", u_eq_slope)
print(f"rank(H_{pe_order}(u_tilde)) = {rank_slope} / {required_slope}")
print(f"Shifted-coordinate reconstruction error: {error_slope:.3e}")

<blockquote style="padding: 18px 20px; margin: 1.2em 0; background: rgba(56, 139, 253, 0.12); border-left: 4px solid rgba(56, 139, 253, 0.85); border-radius: 6px; color: inherit !important;">

##### **Takeaway 2: Persistent excitation is the key data condition**

For the linear Mountain Car, the behavior-spanning property is exact up to numerical precision once the offline input is sufficiently rich. The constant-slope affine case fits the same picture after equilibrium shifting.
</blockquote>

**Reference:** Coulson, Lygeros, and Dörfler, *Data-Enabled Predictive Control: In the Shallows of the DeePC*, 2019.